In [1]:
"""
================================================================================================
NOTEBOOK 03: Ensemble Construction & Statistical Significance Testing
================================================================================================
Author: Mary Wangoi Mwangi (122174)
================================================================================================
Goal:
    - Build a stacking ensemble combining the three trained models from Notebook 02
    - Perform statistical significance testing to validate model differences.

Key Actions:
    1. Load trained models and rebuild data splits from Notebook 02
    2. Build stacking ensemble — RF + XGBoost + LR as base learners
    3. Optimise ensemble threshold using validation set
    4. Perform statistical significance testing (Friedman + Wilcoxon)
    5. Save the ensemble model for final evaluation in Notebook 04

Outcome:
    - Trained stacking ensemble saved to models/ folder.
    - Statistical test results confirming model differences.
    - Ready for final test set evaluation in Notebook 04.
================================================================================================
"""

# ============================================================
# SECTION 1: LIBRARY IMPORTS & GLOBAL CONFIGURATION
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import json
import joblib

from sklearn.model_selection import (StratifiedKFold, cross_val_predict,
                                     cross_validate)
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (recall_score, precision_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay, average_precision_score,
                             precision_recall_curve)
from scipy.stats import wilcoxon, friedmanchisquare
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings('ignore')

# Reproducibility seed
SEED = 42

# Plot styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print("Libraries loaded successfully.")
print(f"   pandas      : {pd.__version__}")
print(f"   numpy       : {np.__version__}")
print(f"   sklearn     : {__import__('sklearn').__version__}")
print(f"   scipy       : {__import__('scipy').__version__}")

Libraries loaded successfully.
   pandas      : 2.3.3
   numpy       : 2.4.3
   sklearn     : 1.8.0
   scipy       : 1.17.1


In [ ]:
# ============================================================
# SECTION 2: LOAD MODELS, METADATA & REBUILD DATA SPLITS
# Load trained models saved from Notebook 02
# Rebuild feature matrix and data splits using same random seed to ensure identical splits as Notebook 02
# ============================================================

# ---- Load trained models ----
best_lr  = joblib.load('../models/best_lr.pkl')
best_rf  = joblib.load('../models/best_rf.pkl')
best_xgb = joblib.load('../models/best_xgb.pkl')

print("Models loaded successfully.")
print(f"   best_lr.pkl  : {type(best_lr).__name__}")
print(f"   best_rf.pkl  : {type(best_rf).__name__}")
print(f"   best_xgb.pkl : {type(best_xgb).__name__}")


# ---- Load metadata ----
with open('../models/model_metadata.json', 'r') as f:
    metadata = json.load(f)

optimal_thresholds = metadata['optimal_thresholds']
cat_cols           = metadata['feature_info']['cat_cols']
num_cols           = metadata['feature_info']['num_cols']

print(f"\n Metadata loaded successfully.")
print(f"   Optimal thresholds:")
for name, thresh in optimal_thresholds.items():
    print(f"      {name:<30} : {thresh}")


# ---- Load cleaned dataset ----
df = pd.read_csv('../data/processed/rta_accidents_labeled.csv')
print(f"\nDataset loaded : {df.shape[0]:,} rows, {df.shape[1]} columns")


# ---- Rebuild feature matrix X and target y ----
# Must use identical preprocessing as Notebook 02
drop_cols = [
    'Severity_Class',
    'Casualty_severity',
    'Casualty_class',
    'Sex_of_casualty',
    'Age_band_of_casualty',
    'Work_of_casuality',
    'Fitness_of_casuality'
]

y = df['Severity_Class']
X = df.drop(columns=drop_cols)


# ---- Feature engineering — identical to Notebook 02 ----
X = X.copy()
X['Hour_of_day']  = pd.to_datetime(X['Time'],
                                    format='%H:%M:%S').dt.hour
X['Is_night']     = X['Hour_of_day'].apply(
    lambda h: 1 if (h >= 20 or h <= 5) else 0)
X['Is_rush_hour'] = X['Hour_of_day'].apply(
    lambda h: 1 if (7 <= h <= 9 or 17 <= h <= 19) else 0)
X['Is_weekend']   = X['Day_of_week'].apply(
    lambda d: 1 if d in ['Saturday', 'Sunday'] else 0)
X = X.drop(columns=['Time'])

print(f"Feature matrix : {X.shape[0]:,} rows, {X.shape[1]} features")


# ---- Rebuild identical 70/15/15 stratified split ----
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED,
    stratify=y_temp
)

print(f"\nData splits rebuilt — identical to Notebook 02:")
print(f"   Training   (70%) : {X_train.shape[0]:,} rows")
print(f"   Validation (15%) : {X_val.shape[0]:,} rows")
print(f"   Test       (15%) : {X_test.shape[0]:,} rows")

print(f"\nClass distribution confirmed:")
print(f"   Train High-Severity : {y_train.sum():,} ({y_train.mean()*100:.1f}%)")
print(f"   Val   High-Severity : {y_val.sum():,}   ({y_val.mean()*100:.1f}%)")
print(f"   Test  High-Severity : {y_test.sum():,}   ({y_test.mean()*100:.1f}%)")

print(f"\nReady for ensemble construction in Cell 3.")